# Assignment 9: Preprocessing, Hyperparameter Tuning & Model Deployment
**Dataset:** California Housing (1990 census), via `sklearn.datasets.fetch_california_housing`
**Goal:** predict the median house value of a block group (in units of $100,000) with a tuned **K-Nearest Neighbours regressor** inside a **pipeline**, then save the pipeline for deployment.

**Workflow**
1. Load and explore the data
2. Split into training and test sets (80/20)
3. Build preprocessing with `ColumnTransformer`
4. Build a `Pipeline` (preprocessing + `KNeighborsRegressor`)
5. Tune `n_neighbors`, `weights` and `p` with `GridSearchCV` and 5-fold cross-validation, scored with **R²**
6. Evaluate the best model on the test set
7. Save the trained pipeline with `joblib` and check that it reloads
8. Bonus: a Streamlit app (`app.py`) that loads the saved pipeline

## 1. Setup

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42
Path("models").mkdir(exist_ok=True)
print("scikit-learn version:", sklearn.__version__)

## 2. Load the dataset
`as_frame=True` returns a pandas DataFrame, which keeps the column names. That matters for the `ColumnTransformer` (we select columns by name) and for the app (it sends a DataFrame with the same names).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame                      # 8 features + target column "MedHouseVal"
FEATURES = list(housing.feature_names)
TARGET = housing.target_names[0]

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

### 2.1 Exploration
Things to look for before choosing the preprocessing:
- **Missing values**: if there are any we would need an imputer.
- **Scale**: `Population` runs into the thousands while `AveBedrms` is around 1. KNN measures **distances**, so unscaled features with big numbers would dominate. **Scaling is essential for KNN.**
- **Skew / outliers**: `Population`, `AveOccup`, `AveRooms` and `AveBedrms` have long right tails (a few block groups have extreme values). A log transform pulls those tails in so that a handful of outliers do not stretch the scale for everyone else.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

print("\nSkewness (|value| > 1 means strongly skewed):")
print(df[FEATURES].skew().round(2).sort_values(ascending=False))

In [ ]:
df.hist(bins=50, figsize=(14, 9))
plt.suptitle("Distributions of features and target", y=1.0)
plt.tight_layout()
plt.show()

Note that the target is **capped at 5.0** ($500,000): the census recorded every more expensive block group as 5.0. No model can predict above that cap, which limits the R² any model can reach.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df["Longitude"], df["Latitude"], c=df[TARGET], cmap="viridis", s=3, alpha=0.5)
plt.colorbar(label="Median house value ($100,000s)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("House values by location: coastal areas are most expensive")
plt.show()

## 3. Train/test split (80/20)
The test set is put aside now and used **once**, at the very end. All tuning uses the training set only.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Training set:", X_train.shape)
print("Test set:    ", X_test.shape)

## 4. Preprocessing with `ColumnTransformer`
| Columns | Steps | Why |
|---|---|---|
| `AveRooms`, `AveBedrms`, `Population`, `AveOccup` (skewed) | `log1p` → `StandardScaler` | shrink extreme values, then put on a common scale |
| `MedInc`, `HouseAge`, `Latitude`, `Longitude` | `StandardScaler` | put on a common scale (mean 0, std 1) |

There are no missing values and no text columns, so no imputer or encoder is needed. (In a production system you might still add a `SimpleImputer` as a safety net for incomplete inputs.)

`np.log1p(x)` = log(1 + x); it is safe for zeros.

In [ ]:
SKEWED = ["AveRooms", "AveBedrms", "Population", "AveOccup"]
REGULAR = ["MedInc", "HouseAge", "Latitude", "Longitude"]

preprocessor = ColumnTransformer([
    ("skewed", Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]), SKEWED),
    ("regular", StandardScaler(), REGULAR),
])

# Quick look at what the preprocessor produces (fitted on the training data only)
preview = pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())
preview.describe().T[["mean", "std", "min", "max"]].round(2)

## 5. Pipeline: preprocessing + KNN
Putting the preprocessor and the model in one `Pipeline` means:
- during cross-validation, the scaler is fitted **only on the training folds**, so no information leaks from the validation fold;
- the saved `.pkl` contains **everything**, so the app can send raw values and get a prediction.

In [ ]:
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", KNeighborsRegressor()),
])
pipeline

### 5.1 Baseline and why scaling matters
Before tuning, compare the default KNN (k = 5) **with** and **without** preprocessing, using 5-fold cross-validation and R².

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

no_scaling = cross_val_score(KNeighborsRegressor(), X_train, y_train, cv=cv, scoring="r2")
with_pipeline = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="r2")

print(f"KNN without preprocessing: R² = {no_scaling.mean():.4f} ± {no_scaling.std():.4f}")
print(f"KNN with pipeline:         R² = {with_pipeline.mean():.4f} ± {with_pipeline.std():.4f}")

## 6. Hyperparameter tuning with `GridSearchCV`
| Hyperparameter | Values | Meaning |
|---|---|---|
| `n_neighbors` | 3, 5, 7, 9 | how many nearby block groups to average |
| `weights` | `uniform`, `distance` | equal votes, or closer neighbours count more |
| `p` | 1, 2 | distance type: 1 = Manhattan, 2 = Euclidean |

Inside a pipeline, parameter names are written `<step>__<parameter>` (double underscore), so `n_neighbors` becomes `model__n_neighbors`.

4 × 2 × 2 = **16 combinations** × 5 folds = **80 model fits**.

In [ ]:
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,              # 5-fold cross-validation
    scoring="r2",       # evaluation metric: R²
    n_jobs=-1,          # use all CPU cores
    return_train_score=True,
)
grid.fit(X_train, y_train)

print("Best parameters:")
for name, value in grid.best_params_.items():
    print(f"   {name.replace('model__', '')}: {value}")
print(f"\nBest cross-validated R²: {grid.best_score_:.4f}")

### 6.1 All 16 combinations, ranked

In [ ]:
results = pd.DataFrame(grid.cv_results_)
table = results[["param_model__n_neighbors", "param_model__weights", "param_model__p",
                 "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]]
table = table.rename(columns={"param_model__n_neighbors": "n_neighbors", "param_model__weights": "weights",
                              "param_model__p": "p", "mean_test_score": "cv_r2", "std_test_score": "cv_r2_std",
                              "mean_train_score": "train_r2", "rank_test_score": "rank"})
table.sort_values("rank").round(4).reset_index(drop=True)

Note: with `weights="distance"`, each training point is its own nearest neighbour (distance 0), so the **training** R² is 1.0. That is expected and is why we judge models on the **cross-validated** score, not the training score.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (w, p), part in results.groupby(["param_model__weights", "param_model__p"]):
    part = part.sort_values("param_model__n_neighbors")
    ax.plot(part["param_model__n_neighbors"], part["mean_test_score"], marker="o",
            label=f"weights={w}, p={p} ({'Manhattan' if p == 1 else 'Euclidean'})")
ax.set_xticks([3, 5, 7, 9])
ax.set_xlabel("n_neighbors"); ax.set_ylabel("Mean cross-validated R²")
ax.set_title("GridSearchCV results")
ax.legend(); plt.tight_layout(); plt.show()

## 7. Evaluate the final model on the test set
`grid.best_estimator_` is the whole pipeline with the best settings, already refitted on the full training set.

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)

print(f"Cross-validated R² (training data): {grid.best_score_:.4f}")
print(f"Test R²:   {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f}  (≈ ${test_rmse * 100_000:,.0f})")
print(f"Test MAE:  {test_mae:.4f}  (≈ ${test_mae * 100_000:,.0f})")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, s=4, alpha=0.4)
plt.plot([0, 5.2], [0, 5.2], "--", color="red", label="perfect prediction")
plt.xlabel("Actual value ($100,000s)"); plt.ylabel("Predicted value ($100,000s)")
plt.title(f"Test set: predicted vs actual (R² = {test_r2:.3f})")
plt.legend(); plt.tight_layout(); plt.show()

**Interpretation (update with your own numbers after running):**
- The test R² is close to the cross-validated R², so the model generalises as expected and there is no sign of leakage.
- R² is the share of the variation in house values the model explains (1.0 would be perfect).
- Errors are largest for expensive areas near the $500,000 cap: the vertical band of points at actual = 5.0 shows the model cannot know how far above the cap those houses really are.

## 8. Save the trained pipeline
We save the **whole pipeline** (preprocessing + tuned KNN) with `joblib`, plus a small JSON file describing it. The app uses the JSON to set sensible ranges for its inputs.

In [ ]:
MODEL_PATH = Path("models/california_knn_pipeline.pkl")
joblib.dump(best_model, MODEL_PATH)

metadata = {
    "model": "KNeighborsRegressor in a Pipeline with ColumnTransformer preprocessing",
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "target": f"{TARGET} (median house value, $100,000s)",
    "best_params": {k.replace("model__", ""): v for k, v in grid.best_params_.items()},
    "cv_r2": round(float(grid.best_score_), 4),
    "test_r2": round(float(test_r2), 4),
    "feature_ranges": {
        c: {"min": float(X_train[c].quantile(0.01)), "max": float(X_train[c].quantile(0.99)),
            "median": float(X_train[c].median())}
        for c in FEATURES
    },
}
Path("models/model_metadata.json").write_text(json.dumps(metadata, indent=2))

print(f"Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")
print("Saved models/model_metadata.json")

### 8.1 Reload and predict, as an application would

In [ ]:
loaded = joblib.load(MODEL_PATH)

new_house = pd.DataFrame([{
    "MedInc": 8.3, "HouseAge": 41, "AveRooms": 7.0, "AveBedrms": 1.0,
    "Population": 320, "AveOccup": 2.5, "Latitude": 37.88, "Longitude": -122.23,
}])[FEATURES]

prediction = loaded.predict(new_house)[0]
print(f"Predicted median house value: {prediction:.3f} → ${prediction * 100_000:,.0f}")

assert np.allclose(loaded.predict(X_test), y_pred), "reloaded model differs!"
print("✔ Reloaded pipeline gives identical predictions")

> **Running in Google Colab?** The files are saved in the temporary Colab session. Download them from the 📁 Files panel on the left (`models/california_knn_pipeline.pkl` and `models/model_metadata.json`), or run the cell below.

In [ ]:
try:
    from google.colab import files   # only exists inside Colab
    files.download("models/california_knn_pipeline.pkl")
    files.download("models/model_metadata.json")
except ImportError:
    print("Not running in Colab: the files are already in the local models/ folder.")

## 9. Bonus: Streamlit app
`app.py` (in the repository next to this notebook) loads `models/california_knn_pipeline.pkl` and lets a user enter the 8 features and get a predicted house value. Run it with:

```bash
streamlit run app.py
```

## 10. Summary
| Step | What was done |
|---|---|
| Preprocessing | `ColumnTransformer`: log + standard scaling for skewed features, standard scaling for the rest |
| Split | 80% train / 20% test, `random_state=42` |
| Model | `KNeighborsRegressor` in a `Pipeline` |
| Tuning | `GridSearchCV` over `n_neighbors` × `weights` × `p` (16 combinations) with 5-fold CV, scored by R² |
| Evaluation | best parameters printed; R², RMSE and MAE on the held-out test set |
| Deployment | pipeline saved with `joblib` and reloaded; Streamlit app for interactive predictions |